# 03 — Data leakage analysis

The dataset is synthetic and was generated for software development, machine learning experimentation, demonstration, and portfolio purposes.

## Target leakage
Using the label (or a deterministic function of the future outcome) as an input. In this dataset that means **`demand_level`** and **`connections_next_hour`**.

`demand_level` is defined from next-hour load vs capacity. Feeding it to a classifier is circular.

## Temporal leakage
Fitting preprocessing on the full series, or training on rows whose `timestamp` is after the evaluation window, lets **future** distributions influence parameters that would not be known at prediction time.

## Historical features (allowed)
`connections_previous_*` and rolling averages are computed from hours **strictly before** `timestamp`. Network snapshot fields describe the hour that **just ended**.

## Why a temporal split
A random `train_test_split` would place October 2025 rows in train and March 2024 rows in test. Imputers and trees would see the future. VERSION 0.1 trains on timestamps through June 2025, validates July–September 2025, and tests October–December 2025.


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
from wifi_tunja_smart_predictor.config import PROCESSED_DATASET_PATH, RAW_DATASET_PATH, SYNTHETIC_DATA_DISCLAIMER, TARGET_COLUMN, LEAKAGE_COLUMNS, MODEL_INPUT_COLUMNS
from wifi_tunja_smart_predictor.data.loader import load_raw_dataset
print(SYNTHETIC_DATA_DISCLAIMER)
path = PROCESSED_DATASET_PATH if PROCESSED_DATASET_PATH.is_file() else RAW_DATASET_PATH
df = load_raw_dataset(path)
df.head()


In [ ]:
assert "demand_level" in LEAKAGE_COLUMNS
assert "connections_next_hour" in LEAKAGE_COLUMNS
overlap = set(MODEL_INPUT_COLUMNS) & set(LEAKAGE_COLUMNS)
print("leakage columns in model inputs (must be empty):", overlap)
print("timestamp is prediction time; sample row:")
df[["timestamp", "connections_previous_hour", "connections_next_hour", TARGET_COLUMN]].head()


In [ ]:
# Correlation of next-hour count with historical lags (allowed) vs the forbidden target itself
num = df[["connections_previous_hour", "average_connections_last_24_hours", "connections_next_hour"]].corr()
num
